# Communication-Aware V2V Perception (Inference-First Kaggle Notebook)

- Repo: https://github.com/mohsenshahverdy/comm-aware-v2v-perception
- Kaggle: https://www.kaggle.com/code/mohsenshahverdi/communication-aware-v2v-perception/edit

This notebook is organized for **inference-first communication approach experiments** using existing checkpoint weights.


## 1. Environment setup


In [ ]:
print("Setting up virtual environment...")
!python -m pip install -U pip virtualenv
!virtualenv /kaggle/working/v2v_env
print("v2v_env created at /kaggle/working/v2v_env")


## 2. Install packages


In [ ]:
!/kaggle/working/v2v_env/bin/python -m pip install -U wrapt
!/kaggle/working/v2v_env/bin/python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!/kaggle/working/v2v_env/bin/python -m pip install "cmake>=3.22" "easydict>=1.9" "tqdm>=4.64" "PyYAML>=6.0" "cython>=0.29.36"
!/kaggle/working/v2v_env/bin/python -m pip install "numpy>=1.24" "scipy>=1.11" "matplotlib>=3.7" "scikit-image>=0.22" "opencv-python>=4.8" "open3d>=0.18" "shapely>=2.0"
!/kaggle/working/v2v_env/bin/python -m pip install "torchviz>=0.0.2" "tensorboardX>=2.6" "einops>=0.7" "timm>=0.9"
!/kaggle/working/v2v_env/bin/python -m pip install --only-binary=:all: cumm-cu118 spconv-cu118


## 2.1 Verify runtime


In [ ]:
!/kaggle/working/v2v_env/bin/python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!/kaggle/working/v2v_env/bin/python -c "import spconv; print('spconv ok')"
!/kaggle/working/v2v_env/bin/python -c "import open3d; print('open3d ok')"
!/kaggle/working/v2v_env/bin/python -c "import yaml; print('yaml ok')"


## 3. Clone repo


In [ ]:
!cd /kaggle/working && git clone https://github.com/mohsenshahverdy/comm-aware-v2v-perception.git


## 4. Build extensions


In [ ]:
%cd /kaggle/working/comm-aware-v2v-perception
!/kaggle/working/v2v_env/bin/python src/utils/setup.py build_ext --inplace


## 5. Check datasets/checkpoint inputs


In [ ]:
!ls -lah /kaggle/input
!find /kaggle/input -maxdepth 2 -type d | head -100


## 5.1 Experiment variables


In [ ]:
from pathlib import Path

REPO_DIR = Path("/kaggle/working/comm-aware-v2v-perception")
ENV_PY = Path("/kaggle/working/v2v_env/bin/python")

DATA_ROOT = Path("/kaggle/input/data-all")
CHECKPOINT_INPUT = Path("/kaggle/input/best-epoch")

TRAIN_DIR = DATA_ROOT / "train"
VALIDATE_DIR = DATA_ROOT / "validate"
CARLA_TEST_DIR = DATA_ROOT / "test/test"
CULVER_TEST_DIR = DATA_ROOT / "test/test_culver_city/test_culver_city"

RUN_ROOT = Path("/kaggle/working/approach_runs")
RUN_ROOT.mkdir(exist_ok=True)

print("REPO_DIR:", REPO_DIR)
print("DATA_ROOT exists:", DATA_ROOT.exists(), DATA_ROOT)
print("CHECKPOINT_INPUT exists:", CHECKPOINT_INPUT.exists(), CHECKPOINT_INPUT)
print("TRAIN_DIR exists:", TRAIN_DIR.exists(), TRAIN_DIR)
print("VALIDATE_DIR exists:", VALIDATE_DIR.exists(), VALIDATE_DIR)
print("CARLA_TEST_DIR exists:", CARLA_TEST_DIR.exists(), CARLA_TEST_DIR)
print("CULVER_TEST_DIR exists:", CULVER_TEST_DIR.exists(), CULVER_TEST_DIR)


## 6. Copy checkpoint + patch config helper


## 6.1 Check checkpoint contents


In [ ]:
!find /kaggle/input/best-epoch -maxdepth 3 -type f | head -50
!find /kaggle/input/best-epoch -name "*.pth" -o -name "config.yaml"


In [ ]:
import shutil
import re
from pathlib import Path


def prepare_approach_run(preset, split_name="carla", for_training=False, run_suffix=None):
    suffix = f"_{run_suffix}" if run_suffix else ""
    run_dir = RUN_ROOT / f"{split_name}_{preset}{suffix}"

    if run_dir.exists():
        shutil.rmtree(run_dir)
    shutil.copytree(CHECKPOINT_INPUT, run_dir)

    src_cfg = REPO_DIR / "src/hypes_yaml/point_pillar_intermediate_V2VAM.yaml"
    dst_cfg = run_dir / "config.yaml"
    shutil.copy(src_cfg, dst_cfg)

    src_presets = REPO_DIR / "src/hypes_yaml/communication_approach_presets.yaml"
    dst_presets = run_dir / "communication_approach_presets.yaml"
    shutil.copy(src_presets, dst_presets)

    if for_training:
        validate_dir = VALIDATE_DIR
    else:
        if split_name == "carla":
            validate_dir = CARLA_TEST_DIR
        elif split_name == "culver":
            validate_dir = CULVER_TEST_DIR
        else:
            raise ValueError(split_name)

    text = dst_cfg.read_text()
    text = re.sub(r"communication_preset:\s*\S+", f"communication_preset: {preset}", text)
    text = re.sub(r'root_dir:\s*["\'].*?["\']', f'root_dir: "{TRAIN_DIR}"', text)
    text = re.sub(r'validate_dir:\s*["\'].*?["\']', f'validate_dir: "{validate_dir}"', text)
    dst_cfg.write_text(text)

    mode = "train" if for_training else "inference"
    print("Prepared:", run_dir)
    print("Preset:", preset)
    print("Split:", split_name)
    print("Mode:", mode)
    print("Suffix:", run_suffix)
    print("Config check:")
    !grep -n "communication_preset\|root_dir:\|validate_dir:" {dst_cfg}
    !ls -lah {run_dir} | head

    return run_dir


def patch_train_config(run_dir, fine_tune_epochs=3):
    import re
    from pathlib import Path

    run_dir = Path(run_dir)
    cfg_path = run_dir / "config.yaml"

    ckpts = list(run_dir.glob("net_epoch*.pth"))
    latest_epoch = 0
    for ck in ckpts:
        m = re.search(r"net_epoch(\d+)\.pth$", ck.name)
        if m:
            latest_epoch = max(latest_epoch, int(m.group(1)))

    fine_tune_epochs = int(fine_tune_epochs)
    final_epoches = latest_epoch + fine_tune_epochs

    text = cfg_path.read_text()
    text = re.sub(r"epoches:\s*\d+", f"epoches: {final_epoches}", text)
    text = re.sub(r"eval_freq:\s*\d+", "eval_freq: 1", text)
    text = re.sub(r"save_freq:\s*\d+", "save_freq: 1", text)
    cfg_path.write_text(text)

    print("latest_epoch:", latest_epoch)
    print("fine_tune_epochs:", fine_tune_epochs)
    print("final train_params.epoches:", final_epoches)
    print("Patched train config:", cfg_path)
    !grep -n "epoches:\|eval_freq:\|save_freq:\|root_dir:\|validate_dir:" {cfg_path}


## 7. Inference runner helper


In [ ]:
def run_inference(run_dir, log_name="inference.log"):
    import subprocess, os

    help_cmd = f"cd {REPO_DIR} && PYTHONPATH={REPO_DIR} {ENV_PY} -m src.tools.inference --help"
    help_out = subprocess.getoutput(help_cmd)
    has_global_sort = "--global_sort_detections" in help_out

    extra = "--global_sort_detections" if has_global_sort else ""

    cmd = f"""
    cd {REPO_DIR} && \
    PYTHONPATH={REPO_DIR} \
    {ENV_PY} -u -m src.tools.inference \
      --model_dir {run_dir} \
      --fusion_method intermediate \
      {extra} \
      2>&1 | tee {run_dir / log_name}
    """
    print("global_sort_supported:", has_global_sort)
    print(cmd)
    return subprocess.call(cmd, shell=True)


def run_training(run_dir, log_name="train.log"):
    import subprocess
    cmd = f"""
    cd {REPO_DIR} && \
    PYTHONPATH={REPO_DIR} \
    {ENV_PY} -u -m src.tools.train \
      --hypes_yaml {Path(run_dir) / 'config.yaml'} \
      --model_dir {run_dir} \
      2>&1 | tee {Path(run_dir) / log_name}
    """
    print(cmd)
    return subprocess.call(cmd, shell=True)


In [ ]:
# Run only if inference fails with pcdet_utils/CUDA extension import error
#!/kaggle/working/v2v_env/bin/python src/pcdet_utils/setup.py build_ext --inplace


In [ ]:
!/kaggle/working/v2v_env/bin/python -m src.tools.inference --help


## 8. Run first CARLA checks (baseline/measurement only)


In [ ]:
approach_order = [
    "baseline_full_communication",
    "measurement_full_communication",
]

for preset in approach_order:
    run_dir = prepare_approach_run(preset, split_name="carla")
    code = run_inference(run_dir, log_name=f"{preset}.log")
    print("Exit code:", code)
    if code != 0:
        raise RuntimeError(f"Failed at {preset}")


## 9. Run CARLA selective baselines (after baseline/measurement pass)


In [ ]:
# Run only after baseline/measurement pass
selective_order = [
    "stress_random_all_features",
    "selective_random_comm_only_10",
    "selective_topk_energy_10",
    "robustness_neighbor_packetloss_20",
]

for preset in selective_order:
    run_dir = prepare_approach_run(preset, split_name="carla")
    code = run_inference(run_dir, log_name=f"{preset}.log")
    print("Exit code:", code)
    if code != 0:
        raise RuntimeError(f"Failed at {preset}")


## 10. Optional Culver runs (after CARLA selective works)


In [ ]:
for preset in ["baseline_full_communication", "measurement_full_communication", "selective_topk_energy_10"]:
    run_dir = prepare_approach_run(preset, split_name="culver")
    code = run_inference(run_dir, log_name=f"{preset}.log")
    print("Exit code:", code)
    if code != 0:
        raise RuntimeError(f"Failed at {preset} on Culver")


## 11. Result inspection


In [ ]:
!find /kaggle/working/approach_runs -name "summary_eval.yaml" -o -name "comm_metrics_epoch.csv" -o -name "comm_metrics_frame.jsonl"


## 12. Plot metrics


In [ ]:
import subprocess
for csv in RUN_ROOT.glob("*/comm_metrics_epoch.csv"):
    print("Plotting:", csv)
    cmd = f"cd {REPO_DIR} && PYTHONPATH={REPO_DIR} {ENV_PY} -m src.tools.plot_comm_metrics --csv {csv}"
    subprocess.call(cmd, shell=True)


## Warning

- `baseline_full_communication`, `measurement_full_communication`, `selective_*` use old checkpoint weights.
- `learned_mask_default` and `repair_feature_reconstruction` need training/fine-tuning before results are meaningful.


## Optional: Training / Fine-tuning

Do not run this section before finishing baseline/measurement/selective inference experiments.


In [ ]:
%cd /kaggle/working/comm-aware-v2v-perception

# Example single-GPU training/fine-tuning
!/kaggle/working/v2v_env/bin/python -u -m src.tools.train \
  --hypes_yaml src/hypes_yaml/point_pillar_intermediate_V2VAM.yaml \
  2>&1 | tee /kaggle/working/training.log


## 9. Selective sweeps (topk vs random comm-only)


In [ ]:
selective_sweep_order = [
    "selective_topk_energy_10_05",
    "selective_topk_energy_10_10",
    "selective_topk_energy_10_25",
    "selective_topk_energy_10_50",
    "selective_random_comm_only_05",
    "selective_random_comm_only_10",
    "selective_random_comm_only_25",
    "selective_random_comm_only_50",
]

for preset in selective_sweep_order:
    run_dir = prepare_approach_run(preset, split_name="carla")
    code = run_inference(run_dir, log_name=f"{preset}.log")
    print("Exit code:", code)
    if code != 0:
        raise RuntimeError(f"Failed at {preset}")


## 10. Build clean summary


In [ ]:
!/kaggle/working/v2v_env/bin/python -m src.tools.build_clean_comm_summary --runs_root /kaggle/working/approach_runs
!python - <<'PY'
import pandas as pd
path='/kaggle/working/approach_runs/clean_summary.csv'
df=pd.read_csv(path)
print(df[["run","strategy","keep_ratio","packet_loss_rate","AP@0.3","AP@0.5","AP@0.7","comm_active_ratio","comm_active_neighbors_ratio","comm_feature_bytes_per_frame","comm_metadata_bytes_per_frame","comm_total_bytes_per_frame","comm_normalized_ratio"]].to_string(index=False))
PY


## 11. Plot communication-performance tradeoffs


In [ ]:
import subprocess, pathlib
run_root=pathlib.Path('/kaggle/working/approach_runs')
for csv in run_root.glob('*/comm_metrics_epoch.csv'):
    cmd=f"cd /kaggle/working/comm-aware-v2v-perception && PYTHONPATH=/kaggle/working/comm-aware-v2v-perception /kaggle/working/v2v_env/bin/python -m src.tools.plot_comm_metrics --csv {csv}"
    subprocess.call(cmd, shell=True)

!find /kaggle/working/approach_runs -name "ap70_vs_comm_ratio.png" -o -name "ap50_vs_comm_ratio.png" -o -name "ap70_vs_total_bytes.png"


## 12. Learned Mask/4 workflow (train first, then inference)


In [ ]:
# Learned Mask: TRAIN first
run_dir_p3_train = prepare_approach_run("learned_mask_default", split_name="carla", for_training=True, run_suffix="train")
patch_train_config(run_dir_p3_train, fine_tune_epochs=3)
code = run_training(run_dir_p3_train, log_name="learned-mask_train.log")
print("Learned-mask train exit:", code)
if code != 0:
    raise RuntimeError("Learned-mask training failed")

# Learned Mask: INFERENCE on CARLA test (separate folder)
run_dir_p3_test = prepare_approach_run("learned_mask_default", split_name="carla", for_training=False, run_suffix="test")
import shutil
for p in Path(run_dir_p3_train).glob("net_epoch*.pth"):
    shutil.copy2(p, run_dir_p3_test)
if (Path(run_dir_p3_train) / "latest.pth").exists():
    shutil.copy2(Path(run_dir_p3_train) / "latest.pth", run_dir_p3_test)

code = run_inference(run_dir_p3_test, log_name="learned_mask_default.log")
print("Learned-mask inference exit:", code)
if code != 0:
    raise RuntimeError("Learned-mask inference failed")


In [ ]:
# Repair Network: start from latest learned-mask checkpoint, TRAIN first
run_dir_p4_train = prepare_approach_run("repair_feature_reconstruction", split_name="carla", for_training=True, run_suffix="train")
for p in Path(run_dir_p3_train).glob("net_epoch*.pth"):
    shutil.copy2(p, run_dir_p4_train)
if (Path(run_dir_p3_train) / "latest.pth").exists():
    shutil.copy2(Path(run_dir_p3_train) / "latest.pth", run_dir_p4_train)

patch_train_config(run_dir_p4_train, fine_tune_epochs=3)
code = run_training(run_dir_p4_train, log_name="repair-network_train.log")
print("Repair-network train exit:", code)
if code != 0:
    raise RuntimeError("Repair-network training failed")

# Repair Network: inference on CARLA test (separate folder)
run_dir_p4_test = prepare_approach_run("repair_feature_reconstruction", split_name="carla", for_training=False, run_suffix="test")
for p in Path(run_dir_p4_train).glob("net_epoch*.pth"):
    shutil.copy2(p, run_dir_p4_test)
if (Path(run_dir_p4_train) / "latest.pth").exists():
    shutil.copy2(Path(run_dir_p4_train) / "latest.pth", run_dir_p4_test)

code = run_inference(run_dir_p4_test, log_name="repair_feature_reconstruction.log")
print("Repair-network inference exit:", code)
if code != 0:
    raise RuntimeError("Repair-network inference failed")
